In [ ]:
import os 
import certifi
import requests
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_agent
from langchain_core.tools import tool
# ========================================
# LOAD ENV VARIABLES
# ========================================

os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")
# ========================================
# LOAD ENV VARIABLES
# ========================================

os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
SEARCH_TOOL = TavilySearchResults(
    max_result = 2 
)
#  llm model 

llm = ChatGoogleGenerativeAI(
    model = "gemini-3.8-flash",
    api_key = GEMINI_API_KEY,
    temprature = 0
)
# frecth the prompt from langchain hub 
from langsmith import Client

client = Client()
prompt = repr(client.pull_prompt("hwchase17/react-multi-input-json",dangerously_pull_public_prompt=True))


import requests
from langchain_core.tools import tool

@tool
def Get_Weather(city: str) -> str:
    """Get the current weather information for a given city."""
    
    url = "https://api.weatherstack.com/current"
    
    params = {
        "access_key": WEATHERSTACK_API_KEY,
        "query": city
    }
    
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    if "error" in data:
        return f"Weather API error: {data['error']['info']}"

    current = data["current"]
    location = data["location"]

    return (
        f"City: {location['name']}\n"
        f"Temperature: {current['temperature']}°C\n"
        f"Weather: {current['weather_descriptions'][0]}\n"
        f"Humidity: {current['humidity']}%"
    )

# ============================================
#     tools 
tools = [SEARCH_TOOL,Get_Weather]
#    Create Agent 

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt = prompt
    
)
response = agent.invoke(
    {"messages":{"role":"user","content":"find best palce in indore for visit and also show the weather of city "}}
)

C:\Users\Om\AppData\Local\Temp\ipykernel_22888\3580947677.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [66]:
result["messages"][2].content

'[{"title": "AI NEWS | Episode 12 | Week of October 5–11, 2026", "url": "https://www.youtube.com/watch?v=5JAYjwE4Gos", "content": "# AI NEWS | Episode 12 | Week of October 5–11, 2026\\n## Channel: David Akpovi\\n21 subscribers\\n0 likes\\n\\n### Description\\n1 view\\nPosted: 2026-10-08\\nArtificial intelligence is becoming more powerful, more accessible, and capable of performing an increasingly wide range of tasks. But as its possibilities expand, one question is becoming more important: how far should we allow these technologies to go, and under what conditions?\\nIn this twelfth episode of AI NEWS, we explore a week in which the AI race is no longer just about model performance. With GPT-6, OpenAI is rethinking how people interact with ChatGPT. Mistral continues to pursue its European ambitions with a one-trillion-parameter model, while Anthropic is working to make certain Claude capabilities more affordable and accessible. [...] However, this expansion of access also brings new ch

Failed to refresh cache entry hwchase17/react-multi-input-json: Connection error caused failure to GET /commits/hwchase17/react-multi-input-json/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/react-multi-input-json/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 11001] getaddrinfo failed)"))'))
Content-Length: None
API Key: 
Failed to refresh cache entry hwchase17/react-multi-input-json: Connection error caused failure to GET /commits/hwchase17/react-multi-input-json/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/react-multi-input-json/latest (Caused by Name

### CUSTOM TOOL

In [57]:
@tool
def Get_Weather(city:str):
    url = "https://api.weatherstack.com/current"
    params = {
        "access_key": WEATHERSTACK_API_KEY,
        "query": "Indore"
    }

    response = requests.get(url, params=params, timeout=10)
    data = response.json()
    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return f"""City: {city}
    Temperature: {data['current']['temperature']}°C
    Weather: {data['current']['weather_descriptions'][0]}
    Humidity: {data['current']['humidity']}%"""


ValueError: Function must have a docstring if description not provided.

In [51]:
Get_Weather("Indore")

'City: Indore\n    Temperature: 32°C\n    Weather: Sunny\n    Humidity: 30%'

In [ ]:
return f"""City: {city}
Temperature: {data['current']['temperature']}°C
Weather: {data['current']['weather_descriptions'][0]}
Humidity: {data['current']['humidity']}%"""
